<a href="https://colab.research.google.com/github/akilaIduwara/SDGP_Project/blob/Akila_Induwara/DataSetTraning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**XGBoost**

In [1]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Step 2: Load and Preprocess Data
# Load data (replace with your actual file path)
try:
    df = pd.read_csv('/content/JWD_L2B_DD_ERTmodel.csv', skiprows=1, header=0)

    # Strip spaces from column names
    df.columns = df.columns.str.strip()

    # Print column names to verify
    print("Columns in the CSV file:", df.columns.tolist())

    # Check if 'Log10Res' column exists
    if 'Log10Res' not in df.columns:
        raise KeyError("Column 'Log10Res' not found in the CSV file. Please check the column names.")

    # Clean invalid resistivity values (remove rows where Log10Res is -9999)
    df = df[df['Log10Res'] != -9999]

    # Convert Log10Res to resistivity (ohm-m)
    df['Resistivity'] = 10 ** df['Log10Res']

    # Create target variable: 1 if resistivity is between 20 and 100 ohm-m, else 0
    df['is_water'] = np.where((df['Resistivity'] >= 20) & (df['Resistivity'] <= 100), 1, 0)

    # Extract features (X, Y, Elev, Dist) and target (is_water)
    features = df[['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']]
    target = df['is_water']

except Exception as e:
    print("Error during data loading or preprocessing:", e)
    exit()

# Step 3: Train XGBoost Model
try:
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, target, test_size=0.2, random_state=42
    )

    # Initialize and train XGBoost classifier
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss'
    )
    model.fit(X_train, y_train)

    # Evaluate model
    y_pred = model.predict(X_test)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

except Exception as e:
    print("Error during model training or evaluation:", e)
    exit()

# Step 4: Identify Water Resources and Display Results
try:
    # Predict water locations in the entire dataset
    df['predicted_water'] = model.predict(features)

    # Filter rows where water is predicted
    water_locations = df[df['predicted_water'] == 1]

    # Display results (Depth = absolute value of Elev)
    results = water_locations[[
        'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev'
    ]].rename(columns={'Elev': 'Depth'})

    results['Depth'] = np.abs(results['Depth'])  # Convert elevation to depth

    print("\nIdentified Water Resources:")
    print(results.head(10).to_string(index=False, justify='left'))  # Display formatted output

    # Save results to CSV
    results.to_csv("identified_water_resources.csv", index=False)
    print("Results saved to identified_water_resources.csv")

except Exception as e:
    print("Error during prediction or result display:", e)
    exit()


Columns in the CSV file: ['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev', 'Log10Res', 'Shading']
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      4253
           1       1.00      1.00      1.00     70098

    accuracy                           1.00     74351
   macro avg       0.98      0.98      0.98     74351
weighted avg       1.00      1.00      1.00     74351


Identified Water Resources:
 X_NAD83UTMz16N  Y_NAD83UTMz16N  Depth
3398734.23      24.64           24.53 
3398734.23      24.64           24.63 
3398734.49      24.64           24.43 
3398734.49      24.64           24.53 
3398734.49      24.64           24.63 
3398734.74      24.64           24.23 
3398734.74      24.64           24.33 
3398734.74      24.64           24.43 
3398734.74      24.64           24.53 
3398734.74      24.64           24.63 
Results saved to identified_water_resources.csv


**Random Forest**

In [2]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Step 2: Load and Preprocess Data
try:
    df = pd.read_csv('/content/JWD_L2B_DD_ERTmodel.csv', skiprows=1, header=0)
    df.columns = df.columns.str.strip()
    print("Columns in the CSV file:", df.columns.tolist())

    if 'Log10Res' not in df.columns:
        raise KeyError("Column 'Log10Res' not found in the CSV file. Please check the column names.")

    df = df[df['Log10Res'] != -9999]
    df['Resistivity'] = 10 ** df['Log10Res']
    df['is_water'] = np.where((df['Resistivity'] >= 20) & (df['Resistivity'] <= 100), 1, 0)

    features = df[['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']]
    target = df['is_water']

except Exception as e:
    print("Error during data loading or preprocessing:", e)
    exit()

# Step 3: Train Random Forest Model with 500 Trees
try:
    X_train, X_test, y_train, y_test = train_test_split(
        features, target, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=500,  # Increased number of trees to 500
        random_state=42,
        class_weight='balanced'
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

except Exception as e:
    print("Error during model training or evaluation:", e)
    exit()

# Step 4: Identify Water Resources and Display Results
try:
    df['predicted_water'] = model.predict(features)
    water_locations = df[df['predicted_water'] == 1]

    results = water_locations[['X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']].rename(columns={'Elev': 'Depth'})
    results['Depth'] = np.abs(results['Depth'])

    print("\nIdentified Water Resources:")
    print(results.head(10).to_string(index=False, justify='left'))

    results.to_csv("identified_water_resources.csv", index=False)
    print("Results saved to identified_water_resources.csv")

except Exception as e:
    print("Error during prediction or result display:", e)
    exit()


Columns in the CSV file: ['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev', 'Log10Res', 'Shading']


<ipython-input-2-79fb58a54e89>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Resistivity'] = 10 ** df['Log10Res']
<ipython-input-2-79fb58a54e89>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['is_water'] = np.where((df['Resistivity'] >= 20) & (df['Resistivity'] <= 100), 1, 0)


Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      4253
           1       1.00      1.00      1.00     70098

    accuracy                           1.00     74351
   macro avg       0.99      0.99      0.99     74351
weighted avg       1.00      1.00      1.00     74351


Identified Water Resources:
 X_NAD83UTMz16N  Y_NAD83UTMz16N  Depth
3398734.23      24.64           24.53 
3398734.23      24.64           24.63 
3398734.49      24.64           24.43 
3398734.49      24.64           24.53 
3398734.49      24.64           24.63 
3398734.74      24.64           24.23 
3398734.74      24.64           24.33 
3398734.74      24.64           24.43 
3398734.74      24.64           24.53 
3398734.74      24.64           24.63 
Results saved to identified_water_resources.csv


**ANN Module**

In [1]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Enable Mixed Precision (if using GPU)
tf.keras.mixed_precision.set_global_policy("mixed_float16")

# Step 2: Load and Preprocess Data Efficiently
try:
    df = pd.read_csv('/content/JWD_L2B_DD_ERTmodel.csv', skiprows=1, header=0)
    df.columns = df.columns.str.strip()
    print("Columns in the CSV file:", df.columns.tolist())

    if 'Log10Res' not in df.columns:
        raise KeyError("Column 'Log10Res' not found in the CSV file. Please check the column names.")

    # Filter invalid values
    df = df[df['Log10Res'] != -9999]
    df['Resistivity'] = 10 ** df['Log10Res']
    df['is_water'] = np.where((df['Resistivity'] >= 20) & (df['Resistivity'] <= 100), 1, 0)

    features = df[['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']]
    target = df['is_water']

except Exception as e:
    print("Error during data loading or preprocessing:", e)
    exit()

# Step 3: Train ANN Deep Learning Model with Large Data Handling
try:
    X_train, X_test, y_train, y_test = train_test_split(
        features, target, test_size=0.2, random_state=42
    )

    # Normalize features
    mean = X_train.mean()
    std = X_train.std()
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    # Convert to TensorFlow datasets for efficiency
    batch_size = 64
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train.values, y_train.values)).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset = tf.data.Dataset.from_tensor_slices((X_test.values, y_test.values)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Define ANN model
    model = keras.Sequential([
        keras.layers.Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
        keras.layers.Dropout(0.3),  # Prevents overfitting
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Early stopping callback to prevent overtraining
    early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    # Train the model
    model.fit(train_dataset, epochs=30, validation_data=test_dataset, callbacks=[early_stopping])

    # Evaluate model
    y_pred = (model.predict(X_test) > 0.5).astype(int)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

except Exception as e:
    print("Error during model training or evaluation:", e)
    exit()

# Step 4: Identify Water Resources and Display Results Efficiently
try:
    # Normalize the full dataset
    df_normalized = (features - mean) / std
    dataset_full = tf.data.Dataset.from_tensor_slices(df_normalized.values).batch(batch_size)

    # Predict water locations using optimized batch inference
    predictions = (model.predict(dataset_full) > 0.5).astype(int)
    df['predicted_water'] = predictions

    # Filter water locations
    water_locations = df[df['predicted_water'] == 1]

    results = water_locations[['X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']].rename(columns={'Elev': 'Depth'})
    results['Depth'] = np.abs(results['Depth'])

    print("\nIdentified Water Resources:")
    print(results.head(10).to_string(index=False, justify='left'))

    results.to_csv("identified_water_resources.csv", index=False)
    print("Results saved to identified_water_resources.csv")

except Exception as e:
    print("Error during prediction or result display:", e)
    exit()


Columns in the CSV file: ['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev', 'Log10Res', 'Shading']


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
4647/4647 ━━━━━━━━━━━━━━━━━━━━ 24s 4ms/step - accuracy: 0.9487 - loss: 0.1545 - val_accuracy: 0.9649 - val_loss: 0.0931
Epoch 2/30
4647/4647 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9615 - loss: 0.0986 - val_accuracy: 0.9704 - val_loss: 0.0752
Epoch 3/30
4647/4647 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9653 - loss: 0.0861 - val_accuracy: 0.9734 - val_loss: 0.0615
Epoch 4/30
4647/4647 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - accuracy: 0.9677 - loss: 0.0794 - val_accuracy: 0.9738 - val_loss: 0.0616
Epoch 5/30
4647/4647 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9695 - loss: 0.0738 - val_accuracy: 0.9775 - val_loss: 0.0566
Epoch 6/30
4647/4647 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9703 - loss: 0.0708 - val_accuracy: 0.9793 - val_loss: 0.0519
Epoch 7/30
4647/4647 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9714 - loss: 0.0681 - val_accuracy: 0.9785 - val_loss: 0.0508
Epoch 8/30
4647/4647 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.9721 - loss: 0

LightGBM (Light Gradient Boosting Machine) **bold text**

In [2]:
pip install lightgbm

In [3]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Step 2: Load and Preprocess Data
try:
    df = pd.read_csv('/content/JWD_L2B_DD_ERTmodel.csv', skiprows=1, header=0)
    df.columns = df.columns.str.strip()
    print("Columns in the CSV file:", df.columns.tolist())

    if 'Log10Res' not in df.columns:
        raise KeyError("Column 'Log10Res' not found in the CSV file. Please check the column names.")

    df = df[df['Log10Res'] != -9999]
    df['Resistivity'] = 10 ** df['Log10Res']
    df['is_water'] = np.where((df['Resistivity'] >= 20) & (df['Resistivity'] <= 100), 1, 0)

    features = df[['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']]
    target = df['is_water']

except Exception as e:
    print("Error during data loading or preprocessing:", e)
    exit()

# Step 3: Train LightGBM Model
try:
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

    # Initialize and train LightGBM model
    model = lgb.LGBMClassifier(
        objective='binary',
        metric='binary_error',
        n_estimators=500,  # Number of trees
        class_weight='balanced'  # Handle class imbalance if needed
    )
    model.fit(X_train, y_train)

    # Evaluate model
    y_pred = model.predict(X_test)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

except Exception as e:
    print("Error during model training or evaluation:", e)
    exit()

# Step 4: Identify Water Resources and Display Results
try:
    # Predict water locations using the trained model
    df['predicted_water'] = model.predict(features)

    # Filter rows where water is predicted
    water_locations = df[df['predicted_water'] == 1]

    # Display results (Depth = absolute value of Elev)
    results = water_locations[['X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']].rename(columns={'Elev': 'Depth'})
    results['Depth'] = np.abs(results['Depth'])  # Convert elevation to depth

    print("\nIdentified Water Resources:")
    print(results.head(10).to_string(index=False, justify='left'))

    # Save results to CSV
    results.to_csv("identified_water_resources.csv", index=False)
    print("Results saved to identified_water_resources.csv")

except Exception as e:
    print("Error during prediction or result display:", e)
    exit()


Columns in the CSV file: ['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev', 'Log10Res', 'Shading']
[LightGBM] [Info] Number of positive: 279732, number of negative: 17670
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011300 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 800
[LightGBM] [Info] Number of data points in the train set: 297402, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.99      0.94      4253
           1       1.00      0.99      1.00     70098

    accuracy                           0.99     74351
   macro avg       0.95      0.99      0.97     74351
weighted avg       0.99      0.99      0.99     74351


Identified Water Resources:
 X_NAD83UTMz16N  Y_NAD83UT

**New ANN Code (Optimised)**

In [1]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Enable Mixed Precision for GPU optimization (if available)
tf.keras.mixed_precision.set_global_policy("mixed_float16")

# Step 2: Load and Preprocess Data Efficiently
try:
    # Load dataset and print column names for debugging
    file_path = '/content/JWD_L2B_DD_ERTmodel.csv'
    df = pd.read_csv(file_path, skiprows=1, header=0)

    print("Dataset Columns:", df.columns.tolist())  # Debugging step

    # Standardized column names
    df.columns = df.columns.str.strip()  # Remove whitespace from column names

    # Expected columns (update if column names are different)
    expected_columns = ['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev', 'Log10Res']

    # Ensure all expected columns exist
    missing_cols = [col for col in expected_columns if col not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing columns in dataset: {missing_cols}. Please check the file.")

    # Filter dataset to required columns
    df = df[expected_columns]

    # Remove invalid resistivity values
    df = df[df['Log10Res'] != -9999].copy()

    # Compute actual resistivity and define water presence
    df['Resistivity'] = np.power(10, df['Log10Res'])
    df['is_water'] = ((df['Resistivity'] >= 20) & (df['Resistivity'] <= 100)).astype(np.uint8)

    # Select features and target
    features = df[['Dist', 'X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']].values
    target = df['is_water'].values

except Exception as e:
    print("Error during data loading or preprocessing:", e)
    exit()

# Step 3: Train ANN Deep Learning Model with Optimized Data Handling
try:
    # Split dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, target, test_size=0.2, random_state=42, stratify=target
    )

    # Normalize features
    mean, std = X_train.mean(axis=0), X_train.std(axis=0)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    # Convert datasets to TensorFlow format
    batch_size = 256
    train_dataset = (
        tf.data.Dataset.from_tensor_slices((X_train.astype(np.float32), y_train))
        .shuffle(10000)
        .batch(batch_size)
        .cache()
        .prefetch(tf.data.AUTOTUNE)
    )
    test_dataset = (
        tf.data.Dataset.from_tensor_slices((X_test.astype(np.float32), y_test))
        .batch(batch_size)
        .cache()
        .prefetch(tf.data.AUTOTUNE)
    )

    # Define ANN model
    model = keras.Sequential([
        keras.layers.Dense(512, input_shape=(X_train.shape[1],)),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),
        keras.layers.Dropout(0.3),

        keras.layers.Dense(256),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),
        keras.layers.Dropout(0.3),

        keras.layers.Dense(128),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(),

        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Early stopping
    early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    # Train the model
    model.fit(train_dataset, epochs=50, validation_data=test_dataset, callbacks=[early_stopping])

    # Evaluate model
    y_pred = (model.predict(X_test, batch_size=batch_size) > 0.5).astype(int)
    print("Classification Report:\n", classification_report(y_test, y_pred))

except Exception as e:
    print("Error during model training or evaluation:", e)
    exit()

# Step 4: Identify Water Resources and Display Results Efficiently
try:
    # Normalize full dataset
    df_normalized = (features - mean) / std
    dataset_full = tf.data.Dataset.from_tensor_slices(df_normalized.astype(np.float32)).batch(batch_size)

    # Predict water locations
    predictions = (model.predict(dataset_full, batch_size=batch_size) > 0.5).astype(np.uint8)
    df['predicted_water'] = predictions

    # Filter identified water locations
    water_locations = df[df['predicted_water'] == 1]

    # Prepare results for export
    results = water_locations[['X_NAD83UTMz16N', 'Y_NAD83UTMz16N', 'Elev']].rename(columns={'Elev': 'Depth'})
    results['Depth'] = np.abs(results['Depth'])

    print("\nIdentified Water Resources:")
    print(results.head(10).to_string(index=False, justify='left'))

    # Save results to CSV
    results.to_csv("identified_water_resources.csv", index=False)
    print("Results saved to identified_water_resources.csv")

except Exception as e:
    print("Error during prediction or result display:", e)
    exit()


Dataset Columns: ['Dist', ' X_NAD83UTMz16N', ' Y_NAD83UTMz16N', ' Elev', ' Log10Res', ' Shading ']


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - accuracy: 0.9468 - loss: 0.1531 - val_accuracy: 0.9543 - val_loss: 0.1160
Epoch 2/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9556 - loss: 0.1139 - val_accuracy: 0.9592 - val_loss: 0.0993
Epoch 3/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9567 - loss: 0.1075 - val_accuracy: 0.9627 - val_loss: 0.0899
Epoch 4/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9580 - loss: 0.1030 - val_accuracy: 0.9655 - val_loss: 0.0864
Epoch 5/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9594 - loss: 0.1004 - val_accuracy: 0.9666 - val_loss: 0.0839
Epoch 6/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9600 - loss: 0.0972 - val_accuracy: 0.9673 - val_loss: 0.0811
Epoch 7/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9609 - loss: 0.0962 - val_accuracy: 0.9678 - val_loss: 0.0778
Epoch 8/50
1162/1162 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9611 - loss: 0.0941 -